In [1]:
"""
Pre-training quality checks on ml_feature_table.csv.
Self-contained: run this whole file top to bottom in one go (or, in a
notebook, Kernel > Restart & Run All) -- don't re-run individual cells
out of order, since the boolean conversion below must run before
anything that touches those columns.
"""

import pandas as pd

FEATURE_COLUMNS = [
    "task_weight", "planned_task_duration_days", "is_planned", "is_cross_department",
    "major_activity_weight", "pct_of_planned_duration_elapsed", "days_remaining_at_prediction",
    "started_late", "num_subtasks_as_of_prediction", "subtask_completion_pct_as_of_prediction",
    "has_subtasks_at_prediction", "num_revisions_before_prediction",
    "position_historical_overdue_rate", "position_has_history",
    "employee_active_workload_at_prediction",
    "department_historical_overdue_rate", "department_recent_overdue_rate_30d",
    "department_has_history",
]
BOOL_COLUMNS = ["is_planned", "is_cross_department", "started_late",
                 "has_subtasks_at_prediction", "position_has_history", "department_has_history"]

# ---- Load ----
df = pd.read_csv("ml_feature_table_for_1_day_after.csv")
print(f"Loaded {len(df)} rows, {df.shape[1]} columns\n")

# ---- Convert Postgres-style boolean text ('t'/'f'/'true'/'false', any case) to real Python bool ----
BOOL_MAP = {"t": True, "f": False, "true": True, "false": False}

def to_bool(series):
    if series.dtype == bool:
        return series  # already converted, nothing to do
    return series.astype(str).str.strip().str.lower().map(BOOL_MAP).astype(bool)

for col in BOOL_COLUMNS:
    if col in df.columns:
        df[col] = to_bool(df[col])

if "target" in df.columns and df["target"].dtype != bool:
    df["target"] = to_bool(df["target"])

# Confirm the conversion actually worked before anything else runs
for col in BOOL_COLUMNS + ["target"]:
    assert df[col].dtype == bool, f"{col} failed to convert -- still {df[col].dtype}"
print("Boolean conversion verified OK for all flag columns.\n")

# ---- 1. Duplicate rows ----
dupes = df["task_id"].duplicated().sum()
print(f"[1] Duplicate task_ids: {dupes}  {'PASS' if dupes == 0 else 'FAIL -- fix the SQL joins before training'}")

# ---- 2. Class balance ----
balance = df["target"].value_counts(normalize=True)
print(f"\n[2] Target class balance:\n{balance}")
minority_pct = balance.min() * 100
print(f"    {'OK' if minority_pct >= 20 else 'WARNING'} -- minority class is {minority_pct:.1f}%")

# ---- 3. Missing values per feature ----
print("\n[3] Missing value % per feature:")
missing_pct = (df[FEATURE_COLUMNS].isnull().mean() * 100).round(1)
shown = missing_pct[missing_pct > 0].sort_values(ascending=False)
print(shown if len(shown) else "    None -- clean")

# ---- 4. Near-zero variance ----
print("\n[4] Feature variance check (flags near-constant features):")
for col in FEATURE_COLUMNS:
    if df[col].dtype == bool or df[col].nunique() <= 2:
        top_share = df[col].value_counts(normalize=True).iloc[0] * 100
        flag = "FLAG -- nearly constant" if top_share > 98 else "ok"
        print(f"    {col}: {top_share:.1f}% one value  [{flag}]")
    elif pd.api.types.is_numeric_dtype(df[col]):
        std = df[col].std()
        flag = "FLAG -- near-zero variance" if (pd.notna(std) and std < 1e-6) else "ok"
        print(f"    {col}: std={std:.4f}  [{flag}]")

# ---- 5. Cold-start / fallback rates (uses == False, not ~, to avoid dtype issues entirely) ----
print("\n[5] Cold-start / fallback rates:")
pos_no_history_pct = (df["position_has_history"] == False).mean() * 100
dept_no_history_pct = (df["department_has_history"] == False).mean() * 100
print(f"    position_has_history = False: {pos_no_history_pct:.1f}% of rows")
print(f"    department_has_history = False: {dept_no_history_pct:.1f}% of rows")
print("    If either is high (>15-20%), that feature is mostly fallback noise for a big chunk of the data.")

# ---- 6. Leakage screen ----
print("\n[6] Correlation with target (screen for leakage -- anything above ~0.9 is suspicious):")
numeric_cols = [c for c in FEATURE_COLUMNS if pd.api.types.is_numeric_dtype(df[c]) or df[c].dtype == bool]
corrs = df[numeric_cols].astype(float).corrwith(df["target"].astype(float)).sort_values(key=abs, ascending=False)
print(corrs.round(3))
suspicious = corrs[corrs.abs() > 0.9]
print(f"    WARNING: {list(suspicious.index)} correlate suspiciously high -- investigate" if len(suspicious)
      else "    No feature above 0.9 -- no obvious direct leakage")

print(f"\n[7] Row count: {len(df)}")

Loaded 13174 rows, 20 columns

Boolean conversion verified OK for all flag columns.

[1] Duplicate task_ids: 0  PASS

[2] Target class balance:
target
False    0.748444
True     0.251556
Name: proportion, dtype: float64
    OK -- minority class is 25.2%

[3] Missing value % per feature:
subtask_completion_pct_as_of_prediction    99.3
dtype: float64

[4] Feature variance check (flags near-constant features):
    task_weight: std=20.5334  [ok]
    planned_task_duration_days: std=7.3357  [ok]
    is_planned: 80.8% one value  [ok]
    is_cross_department: 98.1% one value  [FLAG -- nearly constant]
    major_activity_weight: std=20.3875  [ok]
    pct_of_planned_duration_elapsed: std=0.2825  [ok]
    days_remaining_at_prediction: std=7.3357  [ok]
    started_late: 93.2% one value  [ok]
    num_subtasks_as_of_prediction: std=0.2582  [ok]
    subtask_completion_pct_as_of_prediction: std=26.0587  [ok]
    has_subtasks_at_prediction: 99.3% one value  [FLAG -- nearly constant]
    num_revisions_b